In [61]:
import pandas as pd
import numpy as np
pd.set_option('future.no_silent_downcasting', True)

In [62]:
url_google = "https://docs.google.com/spreadsheets/d/1KXvk6y6WcX-s3b7VZAwCYotDKs3WWY0NvMwccOLpr5Q/export?format=csv&gid=1983317899"
url_facebook = "https://docs.google.com/spreadsheets/d/1KXvk6y6WcX-s3b7VZAwCYotDKs3WWY0NvMwccOLpr5Q/export?format=csv&gid=2025154129"
url_sales="https://docs.google.com/spreadsheets/d/1u7TNCUlMPfIB2SsbjIbmJEnwpVuHo6FD_vQ-vvpCIIc/export?format=csv&gid=472552039"

In [63]:
df_vendas = pd.read_csv(url_sales)
df_vendas.head()

,PedidoSKU,Pedido,Data,Produto,Cor,Tamanho,SKU,Quantidade,Faturamento bruto,Descontos,...,Cotação_USD,Cotação_USD_Corrigida,Custo_Unitario_USD,Custo_Unitario_BRL,Impostos,Taxa de gateway,Taxa de checkout,Custo_BRL,Lucro bruto,Margem de lucro
0,#661503-16349798347-CAPU-BRAN-00G,#661503,24/02/2025,Camisa Polo Ultra,Branco,G,16349798347-CAPU-BRAN-00G,1,"189,9",0,...,"5,73","6,07","12,85","77,99","6,65","7,22","1,14","77,99","105,26","0,55"
1,#661503-16237997367-CAUS-BRAN-00G,#661503,24/02/2025,Camisa Ultra-Stretch,Branco,G,16237997367-CAUS-BRAN-00G,1,"299,9",0,...,"5,73","6,07",0,"99,37","27,29","11,4","1,8","99,37","173,24","0,58"
2,#661513-16349798363-CAPU-AMNH-0GG,#661513,24/02/2025,Camisa Polo Ultra,Azul Marinho,GG,16349798363-CAPU-AMNH-0GG,1,"189,9","28,49",...,"5,73","6,07","12,85","77,99","5,65","6,13","0,97","77,99","77,77","0,48"
3,#661513-16237997369-CAUS-BRAN-0GG,#661513,24/02/2025,Camisa Ultra-Stretch,Branco,GG,16237997369-CAUS-BRAN-0GG,1,"299,9","44,99",...,"5,73","6,07","18,86","114,47","8,92","9,69","1,53","114,47","131,52","0,52"
4,#661513-16349798353-CAPU-VERD-0GG,#661513,24/02/2025,Camisa Polo Ultra,Verde Militar,GG,16349798353-CAPU-VERD-0GG,1,"189,9","28,48",...,"5,73","6,07","12,85","77,99","5,65","6,13","0,97","77,99","77,78","0,48"


In [64]:
colunas_para_converter = ['Faturamento bruto', 'Descontos','Frete preço',
                          'Faturamento líquido','Cotação_USD','Cotação_USD_Corrigida',
                          'Impostos','Taxa de gateway','Taxa de checkout','Custo_BRL']

df_vendas[colunas_para_converter] = df_vendas[colunas_para_converter].apply(lambda col: col.str.replace(',','.',regex=False).astype(float))



In [65]:
df_vendas['Data'] = pd.to_datetime(df_vendas['Data'],dayfirst=True)

In [66]:
df_vendas.columns

Index(['PedidoSKU', 'Pedido', 'Data', 'Produto', 'Cor', 'Tamanho', 'SKU',
       'Quantidade', 'Faturamento bruto', 'Descontos', 'Faturamento líquido',
       'Frete', 'Frete preço', 'cidade', 'estado', 'Estoque', 'Data_TD',
       'Status', 'T&D', 'Motivo_T&D', 'Comentário', 'Dias_entre_datas',
       'Cotação_USD', 'Cotação_USD_Corrigida', 'Custo_Unitario_USD',
       'Custo_Unitario_BRL', 'Impostos', 'Taxa de gateway', 'Taxa de checkout',
       'Custo_BRL', 'Lucro bruto', 'Margem de lucro'],
      dtype='object')

In [67]:
# 🔹 Calcular o total de produtos em cada pedido
df_vendas['Total Itens no Pedido'] = df_vendas.groupby('Pedido')['Quantidade'].transform('sum')

# 🔹 Calcular o total de produtos em cada pedido
df_vendas['Frete Proporcional'] = (df_vendas['Frete preço'] / df_vendas['Total Itens no Pedido']) * df_vendas['Quantidade']



In [68]:
df_agrupado = df_vendas.groupby(['Data', 'Produto'], as_index=False).agg({
    'Quantidade': 'sum',
    'Faturamento bruto':'sum',
    'Descontos':'sum',
    'Faturamento líquido': 'sum',
    'Frete Proporcional':'sum',
    'Cotação_USD':'first',
    'Cotação_USD_Corrigida':'first',
    'Impostos':'sum',
    'Taxa de gateway':'sum',
    'Taxa de checkout':'sum',
    'Custo_BRL':'sum'   
})

# Função para definir 'Produtos Impactados'
def definir_grupo(nome):
    
    if 'Camisa X-Tretch' == nome:
        return 'Camisa X-Tretch'
    elif 'Camisa Ultra-Stretch' in nome:
        return 'Camisa Ultra-Stretch'
    elif 'Camisa Polo Ultra' in nome:
        return 'Camisa Polo Ultra'
    else:
        return 'Todos'


df_agrupado['Produtos Impactados'] = df_agrupado['Produto'].apply(definir_grupo)

df_agrupado.head()

,Data,Produto,Quantidade,Faturamento bruto,Descontos,Faturamento líquido,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,Taxa de gateway,Taxa de checkout,Custo_BRL,Produtos Impactados
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,0.11,0.02,121.00,Todos
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,124.30,19.61,1759.38,Todos
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,324.58,51.27,5403.17,Camisa Polo Ultra
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,408.38,64.46,6293.24,Camisa Ultra-Stretch
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,346.83,54.79,5831.24,Camisa X-Tretch


In [69]:
# TRATAMENTO DOS DADOS GOOGLE ADS

df_google = pd.read_csv(url_google,skiprows=2)

df_google['Day'] = pd.to_datetime(df_google['Day'])

df_google.rename(columns={'Day':'Data'},inplace=True)

df_google.rename(columns={'Campaign':'Campanha',
                          'Cost':'Investimento'},
                          inplace=True)
# TRATAMENTO DOS DADOS FACEBOOK ADS

df_facebook = pd.read_csv(url_facebook)

df_facebook['Dia'] = pd.to_datetime(df_facebook['Dia'])


df_facebook.rename(columns={'Nome da campanha':'Campanha',
                          'Valor usado (BRL)':'Investimento'},
                          inplace=True)

df_facebook.rename(columns={'Dia':'Data'},inplace=True)


df_ads = pd.concat([df_google,df_facebook],axis=0)


In [70]:
# 🔹 Filtrar campanhas que impactam X-Tretch e Ultra-Stretch ao mesmo tempo
df_ads_xu = df_ads[df_ads['Campanha'].str.contains(r'\[X\]') & df_ads['Campanha'].str.contains(r'\[U\]')].copy()

# 🔹 Calcular investimento total dessas campanhas por dia
df_investimento_xu = df_ads_xu.groupby('Data')['Investimento'].sum().reset_index()
df_investimento_xu.rename(columns={'Investimento': 'Investimento XU'}, inplace=True)

# 🔹 Filtrar vendas apenas de X-Tretch e Ultra-Stretch para cada data
df_receita_xu = df_vendas[df_vendas['Produto'].isin(['Camisa X-Tretch', 'Camisa Ultra-Stretch'])].groupby('Data')['Faturamento líquido'].sum().reset_index()
df_receita_xu.rename(columns={'Faturamento líquido': 'Receita XU'}, inplace=True)

# 🔹 Merge para adicionar a receita XU no dataframe de vendas
df_vendas = df_agrupado.merge(df_receita_xu, on='Data', how='left')

# 🔹 Calcular fração de receita para X-Tretch e Ultra-Stretch considerando apenas esses dois produtos
df_vendas['Fração Receita XU'] = df_vendas.apply(
    lambda row: row['Faturamento líquido'] / row['Receita XU'] if row['Produto'] in ['Camisa X-Tretch', 'Camisa Ultra-Stretch'] else 0,
    axis=1)

# 🔹 Merge para adicionar investimento das campanhas XU
df_vendas = df_vendas.merge(df_investimento_xu, on='Data', how='left')

# 🔹 Distribuir investimento apenas para X-Tretch e Ultra-Stretch proporcionalmente à receita desses produtos
df_vendas['Investimento XU Alocado'] = df_vendas['Fração Receita XU'] * df_vendas['Investimento XU'].fillna(0)

In [71]:
df_vendas.head()

,Data,Produto,Quantidade,Faturamento bruto,Descontos,Faturamento líquido,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,Taxa de gateway,Taxa de checkout,Custo_BRL,Produtos Impactados,Receita XU,Fração Receita XU,Investimento XU,Investimento XU Alocado
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,0.11,0.02,121.00,Todos,19872.47,0.000000,191.72,0.000000
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,124.30,19.61,1759.38,Todos,19872.47,0.000000,191.72,0.000000
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,324.58,51.27,5403.17,Camisa Polo Ultra,19872.47,0.000000,191.72,0.000000
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,408.38,64.46,6293.24,Camisa Ultra-Stretch,19872.47,0.540747,191.72,103.672029
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,346.83,54.79,5831.24,Camisa X-Tretch,19872.47,0.459253,191.72,88.047971


In [72]:
# Função para definir 'Produtos Impactados'
def definir_produto(nome):
    tem_x = '[X]' in nome
    tem_u = '[U]' in nome
    tem_p = '[P]' in nome
    
    if tem_x and tem_u: 
        return 'Ultra e X-Tretch'
    elif tem_x:
        return 'Camisa X-Tretch'
    elif tem_u:
        return 'Camisa Ultra-Stretch'
    elif tem_p:
        return 'Camisa Polo Ultra'
    else:
        return 'Todos'

df_ads['Produtos Impactados'] = df_ads['Campanha'].apply(definir_produto)

In [73]:
df_ads = df_ads[['Data','Campanha','Investimento','Produtos Impactados']]

df_ads = df_ads.groupby(['Data','Produtos Impactados'], as_index=False).agg({'Investimento':'sum'})

df_ads.head()

,Data,Produtos Impactados,Investimento
0,2024-11-01,Camisa Polo Ultra,880.95
1,2024-11-01,Camisa Ultra-Stretch,1636.93
2,2024-11-01,Camisa X-Tretch,1222.75
3,2024-11-01,Todos,3572.24
4,2024-11-01,Ultra e X-Tretch,191.72


In [74]:
df_ads_especifico = df_ads[df_ads['Produtos Impactados'] != 'Todos']
df_ads_geral = df_ads[df_ads['Produtos Impactados'] == 'Todos']


In [75]:
df_vendas = df_vendas.merge(df_ads_especifico, how='left', on=['Data','Produtos Impactados'])

In [76]:
#df_vendas.rename(columns={'Investimento':'Investimento Campanhas Específicas'},inplace=True)

In [77]:
df_vendas.head(11)

,Data,Produto,Quantidade,Faturamento bruto,Descontos,Faturamento líquido,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,Taxa de gateway,Taxa de checkout,Custo_BRL,Produtos Impactados,Receita XU,Fração Receita XU,Investimento XU,Investimento XU Alocado,Investimento
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,0.11,0.02,121.00,Todos,19872.47,0.000000,191.72,0.000000,NaN
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,124.30,19.61,1759.38,Todos,19872.47,0.000000,191.72,0.000000,NaN
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,324.58,51.27,5403.17,Camisa Polo Ultra,19872.47,0.000000,191.72,0.000000,880.95
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,408.38,64.46,6293.24,Camisa Ultra-Stretch,19872.47,0.540747,191.72,103.672029,1636.93
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,346.83,54.79,5831.24,Camisa X-Tretch,19872.47,0.459253,191.72,88.047971,1222.75
5,2024-11-01,Camisa X-Tretch Manga Curta,4,719.2,352.05,367.15,3.447302,5.81,6.16,18.89,13.95,2.21,341.64,Todos,19872.47,0.000000,191.72,0.000000,NaN
6,2024-11-01,Camiseta TecModal,15,1948.5,795.42,1153.08,60.054913,5.81,6.16,104.92,43.83,6.94,720.00,Todos,19872.47,0.000000,191.72,0.000000,NaN
7,2024-11-01,Kit 10 Cuecas,1,199.5,53.21,146.29,2.244000,5.81,6.16,5.12,5.56,0.88,110.50,Todos,19872.47,0.000000,191.72,0.000000,NaN
8,2024-11-01,Necessaire de Viagem,16,958.4,348.66,609.74,31.032833,5.81,6.16,21.33,23.16,3.66,0.00,Todos,19872.47,0.000000,191.72,0.000000,NaN
9,2024-11-01,Troca estendida (30 dias),15,60.0,16.24,43.76,36.548778,5.81,6.16,3.97,1.63,0.28,0.00,Todos,19872.47,0.000000,191.72,0.000000,NaN


In [78]:
df_receita_total = df_vendas.groupby('Data')['Faturamento líquido'].sum().reset_index()

df_receita_total.rename(columns={'Faturamento líquido':'Receita Total'},inplace=True)

df_receita_total.head()

,Data,Receita Total
0,2024-11-01,34006.79
1,2024-11-02,28493.48
2,2024-11-03,31612.27
3,2024-11-04,39094.40
4,2024-11-05,35213.12


In [79]:
df_vendas.head()

,Data,Produto,Quantidade,Faturamento bruto,Descontos,Faturamento líquido,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,Taxa de gateway,Taxa de checkout,Custo_BRL,Produtos Impactados,Receita XU,Fração Receita XU,Investimento XU,Investimento XU Alocado,Investimento
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,0.11,0.02,121.00,Todos,19872.47,0.000000,191.72,0.000000,NaN
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,124.30,19.61,1759.38,Todos,19872.47,0.000000,191.72,0.000000,NaN
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,324.58,51.27,5403.17,Camisa Polo Ultra,19872.47,0.000000,191.72,0.000000,880.95
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,408.38,64.46,6293.24,Camisa Ultra-Stretch,19872.47,0.540747,191.72,103.672029,1636.93
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,346.83,54.79,5831.24,Camisa X-Tretch,19872.47,0.459253,191.72,88.047971,1222.75


In [80]:
df_vendas = df_vendas.merge(df_receita_total, on='Data',how='left')

In [81]:
df_vendas['Fração Receita'] = df_vendas['Faturamento líquido'] / df_vendas['Receita Total']

In [82]:
df_vendas.head()

,Data,Produto,Quantidade,Faturamento bruto,Descontos,Faturamento líquido,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,...,Taxa de checkout,Custo_BRL,Produtos Impactados,Receita XU,Fração Receita XU,Investimento XU,Investimento XU Alocado,Investimento,Receita Total,Fração Receita
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,...,0.02,121.00,Todos,19872.47,0.000000,191.72,0.000000,NaN,34006.79,0.000085
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,...,19.61,1759.38,Todos,19872.47,0.000000,191.72,0.000000,NaN,34006.79,0.096177
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,...,51.27,5403.17,Camisa Polo Ultra,19872.47,0.000000,191.72,0.000000,880.95,34006.79,0.251147
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,...,64.46,6293.24,Camisa Ultra-Stretch,19872.47,0.540747,191.72,103.672029,1636.93,34006.79,0.315995
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,...,54.79,5831.24,Camisa X-Tretch,19872.47,0.459253,191.72,88.047971,1222.75,34006.79,0.268373


In [83]:
df_investimento_geral = df_ads_geral.groupby('Data')['Investimento'].sum().reset_index()
df_investimento_geral.rename(columns={'Investimento':'Investimento Geral'},inplace=True)

In [84]:
df_vendas = df_vendas.merge(df_investimento_geral, on='Data',how='left')

In [85]:
df_vendas.head()

,Data,Produto,Quantidade,Faturamento bruto,Descontos,Faturamento líquido,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,...,Custo_BRL,Produtos Impactados,Receita XU,Fração Receita XU,Investimento XU,Investimento XU Alocado,Investimento,Receita Total,Fração Receita,Investimento Geral
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,...,121.00,Todos,19872.47,0.000000,191.72,0.000000,NaN,34006.79,0.000085,3572.24
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,...,1759.38,Todos,19872.47,0.000000,191.72,0.000000,NaN,34006.79,0.096177,3572.24
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,...,5403.17,Camisa Polo Ultra,19872.47,0.000000,191.72,0.000000,880.95,34006.79,0.251147,3572.24
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,...,6293.24,Camisa Ultra-Stretch,19872.47,0.540747,191.72,103.672029,1636.93,34006.79,0.315995,3572.24
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,...,5831.24,Camisa X-Tretch,19872.47,0.459253,191.72,88.047971,1222.75,34006.79,0.268373,3572.24


In [86]:
df_vendas = df_vendas.fillna(0)

df_vendas['Investimento Alocado'] = df_vendas['Investimento XU Alocado'] + df_vendas['Investimento'] + (df_vendas['Fração Receita'] * df_vendas['Investimento Geral'].fillna(0))

df_vendas.rename(columns={'Investimento':'Investimento Campanhas Direcionadas',
                          'Investimento XU Alocado':'Investimento Campanhas XU',
                         'Produtos Impactados':'Foco da Campanha',
                         'Faturamento líquido':'Receita',
                         'Quantidade':'Quantidade vendida'},inplace=True)

In [87]:
df_vendas['Lucro'] = df_vendas['Receita'] + df_vendas['Frete Proporcional'] - df_vendas['Custo_BRL'] - df_vendas['Investimento Alocado'] - df_vendas['Impostos'] - df_vendas['Taxa de gateway'] - df_vendas['Taxa de checkout']
df_vendas['Lucro por peça'] = df_vendas['Lucro'] / df_vendas['Quantidade vendida']

In [88]:
ProductProfitDB = df_vendas.to_csv(r"C:\Users\GabrielCielo\Desktop\consolatio\Databases\ProductProfitDB.csv",float_format="%.2f",index=False,sep=';',decimal=',')

In [89]:
df_vendas.head()

,Data,Produto,Quantidade vendida,Faturamento bruto,Descontos,Receita,Frete Proporcional,Cotação_USD,Cotação_USD_Corrigida,Impostos,...,Fração Receita XU,Investimento XU,Investimento Campanhas XU,Investimento Campanhas Direcionadas,Receita Total,Fração Receita,Investimento Geral,Investimento Alocado,Lucro,Lucro por peça
0,2024-11-01,Calça Neo,1,349.9,347.00,2.90,17.100000,5.81,6.16,0.26,...,0.000000,191.72,0.000000,0.00,34006.79,0.000085,3572.24,0.304630,-101.694630,-101.694630
1,2024-11-01,Calça X-Tretch,24,5517.6,2246.92,3270.68,26.230889,5.81,6.16,142.82,...,0.000000,191.72,0.000000,0.00,34006.79,0.096177,3572.24,343.568267,907.232621,37.801359
2,2024-11-01,Camisa Polo Ultra,74,11832.6,3291.88,8540.72,54.216429,5.81,6.16,298.89,...,0.000000,191.72,0.000000,880.95,34006.79,0.251147,3572.24,1778.109115,738.917314,9.985369
3,2024-11-01,Camisa Ultra-Stretch,60,17394.0,6648.02,10745.98,172.887889,5.81,6.16,638.56,...,0.540747,191.72,103.672029,1636.93,34006.79,0.315995,3572.24,2869.412469,644.815420,10.746924
4,2024-11-01,Camisa X-Tretch,69,13103.1,3976.61,9126.49,143.676968,5.81,6.16,528.27,...,0.459253,191.72,88.047971,1222.75,34006.79,0.268373,3572.24,2269.489239,239.547729,3.471706
